Change tracks a bit, we gonna try running with foviate shrink

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import *

In [ ]:
from albumentations.augmentations.geometric.functional import bboxes_piecewise_affine
from mtrain.neg_mask.crops import get_region_crops, get_largest_bbox, padded_bbox
from mtrain.neg_mask.model.datasets.foviate_shrink import (
    do_foviate_shrink,
    get_foviate_remaps,
)
from mtrain.neg_mask.leveled_cropping import (
    load_crop_level_sample_from_directory,
    make_crop_level_pairs_v2,
)
from tqdm import tqdm
import albumentations as A


def _get_bbox_mask(bb, shape):
    zero = np.zeros(shape)
    zero[bb.y : bb.y2, bb.x : bb.x2] = 1
    return zero


def get_foviated_clean_crops(crop_level_path, full_image_size, crop_size, bbox_pad):
    pre_tfm = A.Compose(
        [
            A.Resize(full_image_size, full_image_size, cv2.INTER_AREA),
            A.PadIfNeeded(
                full_image_size, full_image_size, border_mode=cv2.BORDER_CONSTANT
            ),
        ],
        additional_targets={"bbox_mask": "mask"},
    )

    post_tfm = A.Compose(
        [
            A.Resize(crop_size, crop_size, cv2.INTER_AREA),
            A.PadIfNeeded(crop_size, crop_size, border_mode=cv2.BORDER_CONSTANT),
        ],
        additional_targets={"bbox_mask": "mask"},
    )

    for label in ["other", "trash"]:
        dirs = list((crop_level_path / label).glob("*"))
        for p in dirs:
            if (
                not p.is_dir()
                or not (p / "image.jpg").exists()
                or not (p / "source_dir" / "image.jpg").exists()
            ):
                continue

            try:
                sample = load_crop_level_sample_from_directory(p, full_image_size)
            except Exception as ex:
                print(f"WARN: failed in loading sample at {p.name} cause={ex}")
            img, mask = sample.full_image, sample.full_mask

            res = pre_tfm(
                image=img,
                mask=mask,
                bbox_mask=_get_bbox_mask(
                    padded_bbox(sample.bbox, 10, mask.shape), mask.shape
                ),
            )
            t_image, t_mask, t_bbox_mask = res["image"], res["mask"], res["bbox_mask"]
            t_bb = get_largest_bbox(t_bbox_mask)
            map_x, map_y = get_foviate_remaps(t_image.shape, t_bb, crop_size)
            re_img = cv2.remap(t_image, map_x, map_y, interpolation=cv2.INTER_LINEAR)
            re_mask = cv2.remap(t_mask, map_x, map_y, interpolation=cv2.INTER_LINEAR)

            res = post_tfm(image=re_img, mask=re_mask)
            re_img, re_mask = res["image"], res["mask"]

            yield (t_image, t_mask), (re_img, re_mask), label, p.stem

            # dest_dir = mkdir(root_dest_dir / label / p.name)
            # DiskImage.save(crop, dest_dir / "orig.jpg")
            # DiskBooleanMask.save(mask, dest_dir / "mask.png")

def save_foveated_crops(crop_level_dir, root_dest_dir, full_image_size, crop_size, bbox_pad):
    it = get_foviated_clean_crops(CROP_LEVEL_DIR, full_image_size, crop_size, bbox_pad)
    root_images_dir = mkdir(root_dest_dir / "train")
    root_masks_dir = mkdir(root_dest_dir / "masks")
    for item in tqdm(it):
        (img, mask), (re_img, re_mask), label, name = item
        fname = f"{label}_{name}"
        DiskImage.save(re_img, root_images_dir / f"{fname}.jpg")
        DiskBooleanMask.save(re_mask, root_masks_dir / f"{fname}.png")

In [ ]:
CROP_LEVEL_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level"
)
FOVEATED_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/foveated")

save_foveated_crops(CROP_LEVEL_DIR, FOVEATED_DIR, 1024, 224, 10)

In [ ]:
(img, mask), (re_img, re_mask) = next(it)
show([img, OV(img, mask), re_img, OV(re_img, re_mask)])